# Readability Demo
Colby Le (ncc9kn)

### Goals
- validate the efficacy of the CLEAR dataset for assessing readabililty
- validate the efficacy of two categories of identified readability frameworks:
  1. Objective readability scores (based on syntactic patterns, number of syllables, etc.) - e.x. Flesch Reading Ease. These will come in the form of packages such as `Py-readability-metrics` or `Textstat`, I'm curious if the functions for Flesch Reading Ease in these packages will return a different score than the included Flesch Reading Ease scores in the CLEAR dataset. If not, we can use the included scores as a framework in and of themselves; and if so we could consider the included scores and the function-generated scores as two separate frameworks. Unfortunately, each score has its limitations, for example Flesch Reading Ease can't factor in the lexical difficulty of big vocab words.
  2. LLM-as-a-judge (could prompt ChatGPT 4.0 Turbo to score CLEAR excerpts on a scale of 0 - 100, then compare that with Claude's 0-100 assessments on the same texts). I will not directly do that right now because of API costs but I will instead prompt Claude as a stand-in for the OpenAI model it will later be compared with.

### Assessing readability using the CLEAR dataset
- Contains objective readability scores such as Flesch Reading Ease (0 - 100). Higher scores correlate with an elementary level reading skill, and a lower score correlate with a post-grad reading skill
  - contains both the Flesch Reading Ease score and the grade-level equivalent of the score (Flesch-Kincaid Grade Level)
- read further: 
  - https://seantrott.substack.com/p/measuring-the-readability-of-texts
  - https://www.commonlit.org/blog/introducing-the-clear-corpus-an-open-dataset-to-advance-research-28ff8cfea84a/
  - https://docs.google.com/spreadsheets/d/1sfsZhhP2umXXtmEP_NRErxLuwgN98TyH7LWOq3j07O0/edit?gid=971821388#gid=971821388

### Steps
1. Clean the dataset (only need excerpt text, readability score information)
2. Compare `Py-readability-metrics` and `Textstat` Flesch Reading Ease scores vs. the included dataset scores
3. Prompt Claude to score a subset of excerpts from CLEAR on a scale of 0-100

### Usage of py-readability-metrics
```python
from readability import Readability

r = Readability(text)

r.flesch_kincaid()
r.flesch()
```

```python
fk = r.flesch_kincaid()
print(fk.score)
print(fk.grade_level)
```

```python
f = r.flesch()
print(f.score)
```

### Usage of Textstat
```python
import textstat

textstat.flesch_reading_ease(text)
textstat.flesch_kincaid_grade(text)
```

In [55]:
%%capture
!pip install py-readability-metrics
!pip install textstat
!pip install pandas
!pip install nltk
!pip install anthropic
!python -m nltk.downloader punkt

In [56]:
import pandas as pd
import textstat as t
from readability import Readability as r
import nltk
from anthropic import HUMAN_PROMPT, AI_PROMPT
import anthropic
import os
from dotenv import load_dotenv

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/colby/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/colby/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Step 1: Clean the CLEAR dataset

In [57]:
clear = pd.read_csv('../data/CLEAR_readability_original.csv')
clear.head().T

,0,1,2,3,4
ID,400,401,402,403,404
Last Changed,NaN,NaN,NaN,NaN,NaN
Author,Carolyn Wells,Carolyn Wells,Carolyn Wells,CHARLES KINGSLEY,Charles Kingsley
Title,Patty's Suitors,Two Little Women on a Holiday,Patty Blossom,THE WATER-BABIES\nA Fairy Tale for a Land-Baby,HOW THE ARGONAUTS WERE DRIVEN INTO THE UNKNOWN...
Anthology,NaN,NaN,NaN,NaN,The Heroes\n or Greek Fairy Tales for my...
URL,http://www.gutenberg.org/cache/epub/5631/pg563...,http://www.gutenberg.org/cache/epub/5893/pg589...,http://www.gutenberg.org/cache/epub/20945/pg20...,http://www.gutenberg.org/files/25564/25564-h/2...,http://www.gutenberg.org/files/677/677-h/677-h...
Source,gutenberg,gutenberg,gutenberg,gutenberg,gutenberg
Pub Year,1914.0,1917.0,1917.0,1863.0,1889.0
Category,Lit,Lit,Lit,Lit,Lit
Location,mid,mid,mid,mid,mid


We can use the entirety of the objective scores once we know what they do:
- my extent of understanding: automated readability index and SMOG are rough equivalents to Flesch-Kincaid Grade-Level, they estimate grade-level required to read or years of education required 

```python
clear = clear[['Excerpt', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CARES', 'CML2RI']]

```

In [58]:
clear = clear[['Excerpt', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level']].rename(columns = {'Flesch-Reading-Ease': 'Dataset Flesch Reading Ease', 'Flesch-Kincaid-Grade-Level': 'Dataset Grade Level'})
clear.head()

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level
0,When the young people returned to the ballroom...,81.70,5.95
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86
2,"As Roger had predicted, the snow departed as q...",79.04,6.03
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51
4,And outside before the palace a great garden w...,68.07,12.06


I noticed that the reading ease in this dataset can go as low as -29 and well over 100. Any scale online describes the Flesch Reading Ease as a score between as 0-100, which allows us to translate it into a meaningful Flesch-Kincaid Grade Level. While it is possible to get a Reading Ease lower than 0 and higher than 100, it makes it weird to prompt our LLM later to score on an unbounded scale. Therefore I will drop anything less than 0 or higher than 100, as these seem trivially easy or difficult to read.

In [59]:
flesch_reading_ease_sorted = clear.sort_values(by="Dataset Flesch Reading Ease", ascending=True)[['Excerpt', 'Dataset Flesch Reading Ease']]
flesch_reading_ease_sorted.head()

,Excerpt,Dataset Flesch Reading Ease
2562,"It is further agreed by the parties hereto, th...",-28.99
2057,"What began, during the springtime of my actual...",-25.84
504,Environmental science is an interdisciplinary ...,-21.33
695,Molecular nanotechnology (MNT) is a technology...,-14.79
314,"Lowell was a man of wide learning, and has a p...",-8.59


In [60]:
flesch_reading_ease_sorted = clear.sort_values(by="Dataset Flesch Reading Ease", ascending=False)[['Excerpt', 'Dataset Flesch Reading Ease']]
flesch_reading_ease_sorted.head()

,Excerpt,Dataset Flesch Reading Ease
1327,Cat and Dog walk. They walk in their village. ...,114.03
1328,Cat and Dog open the door. They open the door ...,112.52
3262,"""But why?"" yelped the pup, as the maid threw a...",111.11
3263,"The horse and the cow, in great grief, came an...",109.82
4419,A man tied his horse to a tree and went into a...,108.74


In [61]:
clear = clear[(clear["Dataset Flesch Reading Ease"] >= 0) & (clear["Dataset Flesch Reading Ease"] <= 100)]

flesch_reading_ease_sorted = clear.sort_values(by="Dataset Flesch Reading Ease", ascending=True)[['Excerpt', 'Dataset Flesch Reading Ease']]
print(flesch_reading_ease_sorted.head())
flesch_reading_ease_sorted = clear.sort_values(by="Dataset Flesch Reading Ease", ascending=False)[['Excerpt', 'Dataset Flesch Reading Ease']]
print(flesch_reading_ease_sorted.head())

                                                Excerpt  \
1924  The principal subjects, concerning which Presi...   
2647  An item has appeared recently in several paper...   
821   A social networking service (also social netwo...   
493   An electrostatic generator, or electrostatic m...   
426   Civic technology is technology (mainly informa...   

      Dataset Flesch Reading Ease  
1924                         0.61  
2647                         2.69  
821                          3.05  
493                          3.90  
426                          4.49  
                                                Excerpt  \
3038  One day, John the gardener left a basket of ap...   
1778  I'm planting corn seeds for the new season. I ...   
1741  As it got dark, they saw a light in a house. T...   
2983  "Ah! there goes a butterfly. I will ask him. '...   
1772  Greeny doesn't want to nap. Today, she wants t...   

      Dataset Flesch Reading Ease  
3038                        99.94  
17

# Step 2: Compare Library Functions to included dataset scores

In [62]:
def PYM_flesch_reading_ease(text):
   return r(text).flesch().score

clear["PYM Flesch Reading Ease"] = clear["Excerpt"].apply(PYM_flesch_reading_ease)
clear.head()

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level,PYM Flesch Reading Ease
0,When the young people returned to the ballroom...,81.70,5.95,74.524886
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86,78.415934
2,"As Roger had predicted, the snow departed as q...",79.04,6.03,76.192727
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51,42.368302
4,And outside before the palace a great garden w...,68.07,12.06,62.634463


In [63]:
#There also exists the function r(text).flesch_kincaid().grade_level, but it appears to round the grade level to a whole number, will just use the raw score instead

def PYM_grade_level(text):
   return r(text).flesch_kincaid().score

clear["PYM Grade Level"] = clear["Excerpt"].apply(PYM_grade_level)
clear.head()

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level,PYM Flesch Reading Ease,PYM Grade Level
0,When the young people returned to the ballroom...,81.70,5.95,74.524886,6.744474
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86,78.415934,5.230699
2,"As Roger had predicted, the snow departed as q...",79.04,6.03,76.192727,6.446818
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51,42.368302,19.986478
4,And outside before the palace a great garden w...,68.07,12.06,62.634463,12.343512


In [64]:
def textstat_flesch_reading_ease(text):
    return t.flesch_reading_ease(text)

clear["Textstat Flesch Reading Ease"] = clear["Excerpt"].apply(textstat_flesch_reading_ease)
clear.head()

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level,PYM Flesch Reading Ease,PYM Grade Level,Textstat Flesch Reading Ease
0,When the young people returned to the ballroom...,81.70,5.95,74.524886,6.744474,80.31
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86,78.415934,5.230699,76.11
2,"As Roger had predicted, the snow departed as q...",79.04,6.03,76.192727,6.446818,74.39
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51,42.368302,19.986478,43.06
4,And outside before the palace a great garden w...,68.07,12.06,62.634463,12.343512,72.02


In [65]:
def textstat_grade_level(text):
    return t.flesch_kincaid_grade(text)

clear["Textstat Grade Level"] = clear["Excerpt"].apply(textstat_grade_level)
clear.head()

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level,PYM Flesch Reading Ease,PYM Grade Level,Textstat Flesch Reading Ease,Textstat Grade Level
0,When the young people returned to the ballroom...,81.70,5.95,74.524886,6.744474,80.31,6.1
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86,78.415934,5.230699,76.11,5.6
2,"As Roger had predicted, the snow departed as q...",79.04,6.03,76.192727,6.446818,74.39,6.3
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51,42.368302,19.986478,43.06,20.4
4,And outside before the palace a great garden w...,68.07,12.06,62.634463,12.343512,72.02,11.4


For the sake of maintaining the 0-100 Reading Ease scale, if our other methods also assessed a Reading Ease of < 0 or > 100, we will drop those rows

In [68]:
clear = clear[(clear["PYM Flesch Reading Ease"] >= 0) & (clear["PYM Flesch Reading Ease"] <= 100)]

In [69]:
clear = clear[(clear["Textstat Flesch Reading Ease"] >= 0) & (clear["Textstat Flesch Reading Ease"] <= 100)]

So yes, it appears that depending on the method used you will get a different (but roughly the same) Flesch Reading Ease and Flesch-Kincaid Grade Level

# Step 3: Simulating LLM-as-a-judge as a methodology to assess 0-100 Reading Ease and Approximate Grade Level (on a subset of the excerpts)

potential prompt engineering:
- Should we give the LLM the context that it should approximate the Flesch Reading Ease, or just tell it to give a score on the 0-100 scale?
- Should we give the LLM a verbal or formulaic description of the Flesch Reading Ease process in a sort of Chain-of-Thought? There is the formula 206.835 - 1.015(total words / total sentences) - 84.6(total syllables / total words)
- Should we give the LLM the yielded Dataset Flesch Reading Ease scores for a few examples (Few-Shot Prompting) in hopes it will give more accurate scoring?

In [66]:
#Claude connection
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

class LLM():
    def __init__(self, api_key = api_key, model = "claude-3-5-sonnet-20240620"):
        self.client = anthropic.Anthropic(api_key = api_key)
        self.model = model

    def generate(self, prompt, max_tokens = 1024):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

In [75]:
NUM_SCORED = 5
claude = LLM()

for i in range(NUM_SCORED):
    excerpt = clear.loc[i, 'Excerpt']
    print(f'Excerpt {i}: {excerpt}')
    
    reading_ease_prompt = f'Can you simply output any real number between 0.0 and 100.0 (and only that number) that indicates what you think the reading ease is for the following piece of text: {excerpt}'
    
    claude_reading_ease = claude.generate(reading_ease_prompt)
    clear.loc[i, 'Claude Reading Ease'] = claude_reading_ease

    #this is totally unnecessary but including for curiosity
    grade_level_prompt = f'Can you simply output any real number between 0.0 and 34.0 (and only that number) that indicates what you think the grade level (from elementary to post-grad education) a person would need to read the following piece of text: {excerpt}'
    claude_grade_level = claude.generate(grade_level_prompt)
    clear.loc[i, 'Claude Grade Level'] = claude_grade_level

clear.head(NUM_SCORED + 1)

Excerpt 0: When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.
The floor was covered with snow-white canvas, not laid on smoothly, but rumpled over bumps and hillocks, like a real snow field. The numerous palms and evergreens that had decorated the room, were powdered with flour and strewn with tufts of cotton, like snow. Also diamond dust had been lightly sprinkled on them, and glittering crystal icicles hung from the branches.
At each end of the room, on the wall, hung a beautiful bear-skin rug.
These rugs were for prizes, one for the girls and one for the boys. And this was the game.
The girls were gathered at one end of the room and the boys at the other, and one end was called the North Pole, and the other the South Pole. Each player was given a small flag which they were to plant on reaching the Pole.
This would have been an easy matter, but each traveller was obliged to wear snowsho

,Excerpt,Dataset Flesch Reading Ease,Dataset Grade Level,PYM Flesch Reading Ease,PYM Grade Level,Textstat Flesch Reading Ease,Textstat Grade Level,Claude Reading Ease,Claude Grade Level
0,When the young people returned to the ballroom...,81.70,5.95,74.524886,6.744474,80.31,6.1,72.5,8.2
1,"All through dinner time, Mrs. Fayre was somewh...",80.26,4.86,78.415934,5.230699,76.11,5.6,82.5,8.5
2,"As Roger had predicted, the snow departed as q...",79.04,6.03,76.192727,6.446818,74.39,6.3,73.5,7.2
3,Mr. Grimes was to come up next morning to Sir ...,44.77,20.51,42.368302,19.986478,43.06,20.4,72.5,12.5
4,And outside before the palace a great garden w...,68.07,12.06,62.634463,12.343512,72.02,11.4,72.5,12.5
5,Once upon a time there were Three Bears who li...,80.94,9.47,76.279714,9.629619,75.47,10.0,NaN,NaN


Honestly, the LLM is extremely accurate to the Reading Ease and Grade Level scores for the above five examples...
Next step will be getting OpenAI API access as a comparator framework to judge Claude against

### Upload the cleaned and updated CLEAR dataset

In [76]:
clear.to_csv('../data/CLEAR_readability_test.csv')